# Лабораторная работа №12: Генерация синтетического датасета для VLM

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.

## Цели обучения

После работы студент должен уметь:

- Сгенерировать спецификации сцен и изображения.
- Проверить/отфильтровать VLM и embeddings.
- Сравнить real/synthetic mixtures.


## Теоретическая часть

Корректный pipeline: scene specification → renderer/diffusion → VLM verification → filtering → downstream evaluation. Synthetic test не заменяет реальный test; provenance каждого примера обязателен.


### Математическая постановка

Для смеси $D_\alpha=(1-\alpha)D_{real}+\alpha D_{syn}$ исследуется $Q(\alpha)$ на фиксированном реальном test set. Duplicate filtering использует cosine similarity embeddings.


### Материалы

- [BlenderProc](https://github.com/DLR-RM/BlenderProc)
- [Diffusers](https://huggingface.co/docs/diffusers/index)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.


## Практический протокол

В полном режиме обязательны реальный test-корпус, реальная VLM-проверка синтетики и обучение нескольких downstream-моделей. Процедурный renderer позволяет воспроизводить provenance, но синтетические изображения никогда не называются «реальными». Реальные фотографии и опубликованные метки 2/3 объектов берутся из человечески проверенных validation/test PixMo-Count (ODC-BY-1.0 для датасета; права исходных Flickr-фотографий могут отличаться). Ноутбук автоматически скачивает subset и записывает `data/lab12/real_manifest.jsonl`. Это междоменная оценка переноса счёта геометрических фигур на реальные объекты, а не тест распознавания кругов/квадратов. FAST_DEV_RUN проверяет только renderer и инварианты, не создаёт метрики качества.

In [1]:
from pathlib import Path
import json, os, random, time
import numpy as np
SEED = 42
FAST_DEV_RUN = False  # True: проверка механики на малой выборке, НЕ отчётный эксперимент
ARTIFACTS = Path('artifacts'); ARTIFACTS.mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
def save_json(name, data):
    (ARTIFACTS/name).write_text(json.dumps(data, ensure_ascii=False, indent=2, default=str))

VLM_LOADER = "\n# Одна и та же квантованная модель и интерфейс используются для сравниваемых запусков.\n# В Colab предварительно: %pip install -q 'transformers>=4.51,<5' accelerate bitsandbytes pillow datasets\nimport torch\nfrom PIL import Image\nfrom transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig\nMODEL_ID = os.getenv('CV_VLM_MODEL', 'Qwen/Qwen2.5-VL-3B-Instruct')\nassert torch.cuda.is_available(), 'Требуется GPU для реального VLM-инференса; CPU smoke проверяет только данные и код.'\nquant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)\nprocessor = AutoProcessor.from_pretrained(MODEL_ID)\nmodel = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_ID, quantization_config=quant, device_map='auto').eval()\ndef infer_image(path, prompt, adapter=None):\n    messages = [{'role':'user','content':[{'type':'image','image':Image.open(path).convert('RGB')},{'type':'text','text':prompt}]}]\n    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n    inputs = processor(text=[text], images=[Image.open(path).convert('RGB')], return_tensors='pt').to(model.device)\n    with torch.no_grad(): generated = model.generate(**inputs, max_new_tokens=96, do_sample=False)\n    return processor.batch_decode(generated[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0].strip()\n"

In [2]:
%pip install -q "transformers>=4.51,<5" accelerate bitsandbytes pillow torchvision scikit-learn scipy matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 78.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


## Оценивание

Десять обязательных предметных этапов — по 1 баллу каждый, всего 10 баллов. Отдельное творческое исследование — до 1 дополнительного балла. Каждый этап принимается по указанному наблюдаемому результату.

### Геометрический рендерер

*Вес: 1 балл.*

**Постановка.** Этап «геометрический рендерер» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Реализуйте рендерер цветных фигур без перекрытий с фиксированными координатами.
2. Сохраните и предъявите функция render и словарь палитры.

**Результат.** функция render и словарь палитры.

**Критерий принятия.** Задание считается принятым, если функция render и словарь палитры получены из фактических входов этого этапа и доступны для проверки.

In [3]:
import hashlib, inspect, re, sys, shutil
from collections import Counter, defaultdict
from PIL import Image, ImageDraw
from scipy import ndimage

IMG_SIZE = 224
PALETTE = {'red': (220, 50, 47), 'green': (38, 139, 60), 'blue': (38, 90, 200), 'yellow': (235, 200, 30)}
BACKGROUNDS = {'white': (255, 255, 255), 'gray': (235, 235, 235)}
LAYOUT_Y = [64, 112, 160]                                  # три фиксированные «строки» расположения
SLOT_X = {1: [112], 2: [70, 154], 3: [56, 112, 168]}      # фиксированные x-координаты слотов для 1/2/3 объектов
RADII = {'small': 18, 'large': 24}                        # половина стороны bounding box

def shape_boxes(spec):
    """Фиксированные bounding box объектов: координаты определяются только спецификацией (без случайности)."""
    r, y = RADII[spec['size']], LAYOUT_Y[spec['layout']]
    return [(x - r, y - r, x + r, y + r) for x in SLOT_X[spec['count']]]

def render(spec):
    """Рендерер цветных фигур одного вида и цвета без перекрытий; одинаковая спецификация -> одинаковые пиксели."""
    img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), BACKGROUNDS[spec['bg']]); d = ImageDraw.Draw(img)
    color = PALETTE[spec['color']]
    for x0, y0, x1, y1 in shape_boxes(spec):
        if spec['shape'] == 'circle': d.ellipse([x0, y0, x1, y1], fill=color)
        elif spec['shape'] == 'square': d.rectangle([x0, y0, x1, y1], fill=color)
        elif spec['shape'] == 'triangle': d.polygon([((x0 + x1) / 2, y0), (x0, y1), (x1, y1)], fill=color)
        else: raise ValueError(spec['shape'])
    return img

def pixel_hash(img):
    img = img.convert('RGB')
    return hashlib.sha256(str(img.size).encode() + img.tobytes()).hexdigest()

# --- инварианты рендерера: нет перекрытий, всё внутри кадра, число связных компонент цвета = число фигур
def boxes_disjoint(boxes, gap=2):
    return all(a[2] + gap <= b[0] or b[2] + gap <= a[0] or a[3] + gap <= b[1] or b[3] + gap <= a[1]
               for i, a in enumerate(boxes) for b in boxes[i + 1:])
for n in (1, 2, 3):
    for y in range(3):
        for sz in RADII:
            bx = shape_boxes(dict(count=n, layout=y, size=sz))
            assert boxes_disjoint(bx) and all(0 <= b[0] and b[2] < IMG_SIZE and 0 <= b[1] and b[3] < IMG_SIZE for b in bx)
cols = list(PALETTE.values())
assert min(sum((a - b) ** 2 for a, b in zip(c1, c2)) ** .5 for i, c1 in enumerate(cols) for c2 in cols[i + 1:]) > 100   # цвета различимы

demo = [dict(shape=s, color='red', count=3, layout=1, size='large', bg='white') for s in ('circle', 'square', 'triangle')]
for sp in demo:
    arr = np.array(render(sp)); mask = (np.abs(arr.astype(int) - np.array(PALETTE['red'])).sum(2) < 30)
    n_comp = ndimage.label(mask)[1]
    assert n_comp == sp['count'], (sp['shape'], n_comp)
    assert pixel_hash(render(sp)) == pixel_hash(render(dict(sp)))                         # детерминизм
print('renderer OK: фигуры не перекрываются, компонент =', 3, '| палитра:', list(PALETTE))

renderer OK: фигуры не перекрываются, компонент = 3 | палитра: ['red', 'green', 'blue', 'yellow']


### Пространство визуальных сцен

*Вес: 1 балл.*

**Постановка.** Этап «пространство визуальных сцен» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Сгенерируйте воспроизводимые сочетания формы, цвета и числа объектов.
2. Сохраните и предъявите воспроизводимые спецификации фигур и количества.

**Результат.** воспроизводимые спецификации фигур и количества.

**Критерий принятия.** Задание считается принятым, если воспроизводимые спецификации фигур и количества получены из фактических входов этого этапа и доступны для проверки.

In [4]:
SHAPES, COLORS, COUNTS = ['circle', 'square', 'triangle'], list(PALETTE), [1, 2, 3]
LAYOUTS, SIZES, BGS = [0, 1, 2], list(RADII), list(BACKGROUNDS)
SPACE = [dict(shape=s, color=c, count=n, layout=l, size=z, bg=b)
         for s in SHAPES for c in COLORS for n in COUNTS for l in LAYOUTS for z in SIZES for b in BGS]
N_SYN = 24 if FAST_DEV_RUN else 240
rng = np.random.RandomState(SEED)
picks = rng.randint(0, len(SPACE), N_SYN)                 # выборка С возвращением: дубли возможны и устраняются на этапе дедупликации
specs = [{'spec_id': f'syn-{i:04d}', 'space_index': int(k), **SPACE[k]} for i, k in enumerate(picks)]
assert len({json.dumps(SPACE[k], sort_keys=True) for k in range(len(SPACE))}) == len(SPACE)
assert {s['count'] for s in specs} <= set(COUNTS)

def make_question(spec):
    return f"How many {spec['color']} {spec['shape']}s are visible? Answer with one integer."

save_json('scene_specs.json', {'seed': SEED, 'space_size': len(SPACE), 'n_sampled': N_SYN, 'sampling': 'uniform with replacement',
                               'axes': {'shape': SHAPES, 'color': COLORS, 'count': COUNTS, 'layout': LAYOUTS, 'size': SIZES, 'bg': BGS},
                               'count_distribution': dict(Counter(s['count'] for s in specs)), 'specs': specs})
print('пространство сцен:', len(SPACE), '| выбрано:', N_SYN, '| по числу объектов:', dict(sorted(Counter(s['count'] for s in specs).items())))
print('пример:', specs[0], '->', make_question(specs[0]))

пространство сцен: 432 | выбрано: 240 | по числу объектов: {1: 65, 2: 88, 3: 87}
пример: {'spec_id': 'syn-0000', 'space_index': 102, 'shape': 'circle', 'color': 'blue', 'count': 3, 'layout': 1, 'size': 'large', 'bg': 'white'} -> How many blue circles are visible? Answer with one integer.


### Изображения и вопросы

*Вес: 1 балл.*

**Постановка.** Этап «изображения и вопросы» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Сохраните изображения, вопрос и ответ для каждой сцены.
2. Сохраните и предъявите файлы изображений и связанные с ними ответы.

**Результат.** файлы изображений и связанные с ними ответы.

**Критерий принятия.** Задание считается принятым, если файлы изображений и связанные с ними ответы получены из фактических входов этого этапа и доступны для проверки.

In [5]:
SYN_DIR = Path('data/lab12/synthetic'); SYN_DIR.mkdir(parents=True, exist_ok=True)
synthetic_records = []
for sp in specs:
    img = render(sp); path = (SYN_DIR / f"{sp['spec_id']}.png").resolve(); img.save(path)
    synthetic_records.append({'id': sp['spec_id'], 'image': str(path), 'question': make_question(sp),
                              'answer': str(sp['count']), 'image_sha256': pixel_hash(img),
                              'spec': {k: sp[k] for k in ('shape', 'color', 'count', 'layout', 'size', 'bg')}, 'space_index': sp['space_index']})
assert all(Path(r['image']).is_file() and pixel_hash(Image.open(r['image'])) == r['image_sha256'] for r in synthetic_records)   # PNG без потерь
assert all(r['answer'] in ('1', '2', '3') and r['question'] and r['spec']['shape'] in r['question'] for r in synthetic_records)
print('изображений:', len(synthetic_records), '| размер:', Image.open(synthetic_records[0]['image']).size, '| ответы:', dict(Counter(r['answer'] for r in synthetic_records)))

изображений: 240 | размер: (224, 224) | ответы: {'3': 87, '2': 88, '1': 65}


### Происхождение синтетических данных

*Вес: 1 балл.*

**Постановка.** Этап «происхождение синтетических данных» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Зафиксируйте seed, спецификацию, пути и provenance в манифесте.
2. Сохраните и предъявите artifacts/synthetic_manifest.jsonl.

**Результат.** artifacts/synthetic_manifest.jsonl.

**Критерий принятия.** Задание считается принятым, если artifacts/synthetic_manifest.jsonl получены из фактических входов этого этапа и доступны для проверки.

In [6]:
def _src(*fns):
    try: return ''.join(inspect.getsource(f) for f in fns)
    except (OSError, TypeError): return 'source-unavailable'
RENDERER_SHA256 = hashlib.sha256((_src(render, shape_boxes) + json.dumps([PALETTE, BACKGROUNDS, LAYOUT_Y, SLOT_X, RADII], sort_keys=True)).encode()).hexdigest()

synthetic_manifest = []
for r in synthetic_records:
    synthetic_manifest.append({**r, 'split': 'synthetic', 'origin': 'synthetic', 'is_real': False,
                               'generation': {'method': 'procedural PIL renderer (not a photograph)', 'renderer_sha256': RENDERER_SHA256,
                                              'seed': SEED, 'sampling': 'uniform with replacement from the full spec grid'},
                               'license': 'generated by course code; contains no third-party content'})
with (ARTIFACTS / 'synthetic_manifest.jsonl').open('w') as f:
    for r in synthetic_manifest: f.write(json.dumps(r, ensure_ascii=False) + '\n')

# воспроизводимость: из манифеста картинка пересоздаётся побитно
for r in synthetic_manifest:
    assert pixel_hash(render({**r['spec']})) == r['image_sha256'], r['id']
assert len({r['id'] for r in synthetic_manifest}) == len(synthetic_manifest) and not any(r['is_real'] for r in synthetic_manifest)
print('synthetic_manifest.jsonl:', len(synthetic_manifest), 'записей | renderer sha256:', RENDERER_SHA256[:12], '| все воспроизводятся из spec')

synthetic_manifest.jsonl: 240 записей | renderer sha256: 463627521be0 | все воспроизводятся из spec


### Ответы VLM и визуальная дедупликация

*Вес: 1 балл.*

**Постановка.** Этап «ответы vlm и визуальная дедупликация» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Спросите реальную VLM о числе фигур и найдите дубли по хешу пикселей.
2. Сохраните и предъявите artifacts/verification.jsonl с реальными ответами VLM.

**Результат.** artifacts/verification.jsonl с реальными ответами VLM.

**Критерий принятия.** Задание считается принятым, если artifacts/verification.jsonl с реальными ответами VLM получены из фактических входов этого этапа и доступны для проверки.

In [7]:
HAVE_VLM = not FAST_DEV_RUN                       # FAST_DEV_RUN: VLM не запускается, метрики не создаются
VER_PATH = ARTIFACTS / 'verification.jsonl'
NUM_WORDS = {w: i for i, w in enumerate('zero one two three four five six seven eight nine ten'.split())}

def load_jsonl(path):
    path = Path(path)
    return [json.loads(s) for s in path.read_text().splitlines() if s.strip()] if path.exists() else []

def parse_count(text):
    """Первое целое число в ответе (цифрами или словом); иначе None - такой ответ считается несогласованным."""
    m = re.search(r'\b(\d+)\b', text)
    if m: return int(m.group(1))
    for w in re.findall(r'[a-z]+', text.lower()):
        if w in NUM_WORDS: return NUM_WORDS[w]
    return None
assert parse_count('3') == 3 and parse_count('There are two circles.') == 2 and parse_count('many') is None

# дедупликация по хешу пикселей: первое изображение с данным хешем остаётся, остальные помечаются дублями
first_by_hash, duplicate_of = {}, {}
for r in synthetic_manifest:
    rep = first_by_hash.setdefault(r['image_sha256'], r['id'])
    duplicate_of[r['id']] = None if rep == r['id'] else rep
n_dup = sum(v is not None for v in duplicate_of.values())
save_json('dedup.json', {'n_images': len(synthetic_manifest), 'n_unique': len(first_by_hash), 'n_duplicates': n_dup,
                         'method': 'sha256 пиксельных данных (точные дубли)', 'duplicate_of': {k: v for k, v in duplicate_of.items() if v}})
print(f'уникальных изображений: {len(first_by_hash)} из {len(synthetic_manifest)}; дублей: {n_dup}')

if HAVE_VLM:
    exec(VLM_LOADER, globals())                   # processor, model, infer_image (Qwen2.5-VL-3B, NF4)
    done = {r['id'] for r in load_jsonl(VER_PATH)}
    with VER_PATH.open('a') as f:
        for k, r in enumerate([r for r in synthetic_manifest if r['id'] not in done], 1):
            t0 = time.time(); resp = infer_image(r['image'], r['question'])      # VLM видит только изображение и вопрос
            parsed = parse_count(resp)
            f.write(json.dumps({'id': r['id'], 'response': resp, 'parsed': parsed, 'gold': int(r['answer']),
                                'agrees': parsed == int(r['answer']), 'image_sha256': r['image_sha256'], 'duplicate_of': duplicate_of[r['id']],
                                'seconds': round(time.time() - t0, 3), 'model': MODEL_ID, 'timestamp': time.strftime('%Y-%m-%dT%H:%M:%S')},
                               ensure_ascii=False) + '\n'); f.flush()
            if k % 50 == 0: print('  ', k)
    ver = load_jsonl(VER_PATH)
    assert {r['id'] for r in ver} == {r['id'] for r in synthetic_manifest}
    print('ответов VLM:', len(ver), '| согласны с эталоном:', sum(r['agrees'] for r in ver), '| не распознано:', sum(r['parsed'] is None for r in ver))
else:
    print('FAST_DEV_RUN: VLM-проверка пропущена (нужен GPU); verification.jsonl не создаётся.')

уникальных изображений: 177 из 240; дублей: 63


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/3.53G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (3420) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


   50
   100
   150
   200
ответов VLM: 240 | согласны с эталоном: 240 | не распознано: 0


### Отбор согласованных пар

*Вес: 1 балл.*

**Постановка.** Этап «отбор согласованных пар» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Отберите только изображения с согласованным ответом и без дублей.
2. Сохраните и предъявите artifacts/accepted.jsonl после проверки ответов.

**Результат.** artifacts/accepted.jsonl после проверки ответов.

**Критерий принятия.** Задание считается принятым, если artifacts/accepted.jsonl после проверки ответов получены из фактических входов этого этапа и доступны для проверки.

In [8]:
if HAVE_VLM:
    ver = {r['id']: r for r in load_jsonl(VER_PATH)}
    accepted, reasons = [], Counter()
    for r in synthetic_manifest:
        v = ver[r['id']]
        if v['duplicate_of']: reasons['duplicate'] += 1; continue
        if v['parsed'] is None: reasons['unparsable'] += 1; continue
        if not v['agrees']: reasons['vlm_disagrees'] += 1; continue
        accepted.append({**r, 'vlm_response': v['response'], 'vlm_parsed': v['parsed'], 'verified_by': v['model']}); reasons['accepted'] += 1
    with (ARTIFACTS / 'accepted.jsonl').open('w') as f:
        for r in accepted: f.write(json.dumps(r, ensure_ascii=False) + '\n')
    by_count = {c: {'total': sum(r['answer'] == str(c) for r in synthetic_manifest if not ver[r['id']]['duplicate_of']),
                    'accepted': sum(r['answer'] == str(c) for r in accepted)} for c in COUNTS}
    save_json('accept_stats.json', {'reasons': dict(reasons), 'by_count': by_count, 'n_accepted': len(accepted),
                                    'note': 'фильтр по согласию VLM убирает трудные для VLM сцены: оставшийся набор смещён к лёгким примерам'})
    print(dict(reasons)); print('принято по числу фигур (из уникальных):', by_count)
    for c in (2, 3): assert sum(r['answer'] == str(c) for r in accepted) >= 12, f'мало принятых сцен с {c} объектами для смесей'
    assert all(r['vlm_parsed'] == int(r['answer']) and not duplicate_of[r['id']] for r in accepted)
else:
    accepted = []; print('FAST_DEV_RUN: отбор не выполняется (нет ответов VLM).')

{'accepted': 177, 'duplicate': 63}
принято по числу фигур (из уникальных): {1: {'total': 46, 'accepted': 46}, 2: {'total': 66, 'accepted': 66}, 3: {'total': 65, 'accepted': 65}}


### Независимый набор реальных изображений

*Вес: 1 балл.*

**Постановка.** Этап «независимый набор реальных изображений» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Скачайте PixMo-Count validation для обучения и официальный test для отложенной оценки; проверьте метки и отсутствие общих ID/изображений.
2. Сохраните и предъявите проверенный real train/test manifest с источником, split и контрольными суммами.

**Результат.** проверенный real train/test manifest с источником, split и контрольными суммами.

**Критерий принятия.** Задание считается принятым, если проверенный real train/test manifest получены из фактических входов этого этапа и доступны для проверки.

In [9]:
RUN_FULL = not FAST_DEV_RUN
if RUN_FULL:
    for _c in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (_c / 'open_real_scenes.py').exists(): sys.path.insert(0, str(_c)); break
    from open_real_scenes import load_real_scenes        # PixMo-Count: validation -> real train, official test -> real test
    N_REAL_TRAIN, N_REAL_TEST = 24, 12
    real = load_real_scenes('data/lab12', train_size=N_REAL_TRAIN, test_size=N_REAL_TEST)
    real_train = [r for r in real if r['split'] == 'train']; real_test = [r for r in real if r['split'] == 'test']
    assert len(real_train) == N_REAL_TRAIN and len(real_test) == N_REAL_TEST
    for name, part in (('train', real_train), ('test', real_test)):
        assert {r['answer'] for r in part} == {'2', '3'} and len(Counter(r['answer'] for r in part)) == 2
        assert all(Path(r['image']).is_file() and Image.open(r['image']).size[0] > 0 for r in part)
    assert not {r['id'] for r in real_train} & {r['id'] for r in real_test}
    assert not {r['image_sha256'] for r in real_train} & {r['image_sha256'] for r in real_test}
    assert not {r['downloaded_sha256'] for r in real_train} & {r['downloaded_sha256'] for r in real_test}
    assert not {r['source_row'] for r in real_train if r['source_split'] == 'validation'} & {r['source_row'] for r in real_test if r['source_split'] == 'validation'}
    manifest_path = Path('data/lab12/real_manifest.jsonl')
    assert len(manifest_path.read_text().splitlines()) == len(real)
    shutil.copy(manifest_path, ARTIFACTS / 'real_manifest.jsonl')
    save_json('real_manifest_check.json', {
        'n_train': len(real_train), 'n_test': len(real_test), 'classes_train': dict(Counter(r['answer'] for r in real_train)),
        'classes_test': dict(Counter(r['answer'] for r in real_test)), 'source': 'allenai/pixmo-count (validation -> train, test -> test)',
        'dataset_license': 'ODC-BY-1.0; image rights may differ (third-party hosted)',
        'checksum_mismatch': sum(not r['checksum_matches'] for r in real), 'overlap_ids': 0, 'overlap_sha256': 0,
        'note': 'real test очень мал (12 изображений): точность имеет шаг 1/12'})
    print('real train:', Counter(r['answer'] for r in real_train), '| real test:', Counter(r['answer'] for r in real_test),
          '| не совпала контрольная сумма:', sum(not r['checksum_matches'] for r in real))
else:
    real_train, real_test = [], []; print('FAST_DEV_RUN: загрузка реальных изображений пропущена.')

real train: Counter({'2': 12, '3': 12}) | real test: Counter({'3': 6, '2': 6}) | не совпала контрольная сумма: 32


### Признаки реальных и синтетических сцен

*Вес: 1 балл.*

**Постановка.** Этап «признаки реальных и синтетических сцен» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Вычислите одинаковые замороженные признаки ResNet18 для real train, synthetic и real test.
2. Сохраните и предъявите замороженные признаки ResNet18 для трёх наборов.

**Результат.** замороженные признаки ResNet18 для трёх наборов.

**Критерий принятия.** Задание считается принятым, если замороженные признаки ResNet18 для трёх наборов получены из фактических входов этого этапа и доступны для проверки.

In [10]:
if RUN_FULL:
    import torch
    from torchvision import models, transforms
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    resnet.fc = torch.nn.Identity(); resnet.eval().to(device)
    for p_ in resnet.parameters(): p_.requires_grad_(False)                 # один и тот же frozen extractor для всех наборов
    tfm = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(),
                              transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    @torch.no_grad()
    def extract(paths, bs=32):
        out = []
        for k in range(0, len(paths), bs):
            x = torch.stack([tfm(Image.open(p).convert('RGB')) for p in paths[k:k + bs]]).to(device)
            out.append(resnet(x).float().cpu().numpy())
        return np.concatenate(out)
    F_real_train, F_syn, F_real_test = (extract([r['image'] for r in part]) for part in (real_train, accepted, real_test))
    y_real_train, y_syn, y_real_test = (np.array([int(r['answer']) for r in part]) for part in (real_train, accepted, real_test))
    assert F_real_train.shape[1] == F_syn.shape[1] == F_real_test.shape[1] == 512
    np.savez_compressed(ARTIFACTS / 'resnet18_features_lab12.npz', real_train=F_real_train, synthetic=F_syn, real_test=F_real_test,
                        y_real_train=y_real_train, y_synthetic=y_syn, y_real_test=y_real_test,
                        ids_real_train=np.array([r['id'] for r in real_train]), ids_synthetic=np.array([r['id'] for r in accepted]),
                        ids_real_test=np.array([r['id'] for r in real_test]))
    print('признаки:', F_real_train.shape, F_syn.shape, F_real_test.shape, '| device', device)
else:
    print('FAST_DEV_RUN: признаки не считаются.')

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 170MB/s]


признаки: (24, 512) (177, 512) (12, 512) | device cuda


### Обучение на смесях

*Вес: 1 балл.*

**Постановка.** Этап «обучение на смесях» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. При постоянном бюджете train обучите классификаторы с четырьмя долями синтетики.
2. Сохраните и предъявите результаты классификаторов при четырёх долях синтетики.

**Результат.** результаты классификаторов при четырёх долях синтетики.

**Критерий принятия.** Задание считается принятым, если результаты классификаторов при четырёх долях синтетики получены из фактических входов этого этапа и доступны для проверки.

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
if RUN_FULL:
    ALPHAS = [0.0, 0.25, 0.5, 1.0]                      # доля синтетики в обучающей смеси
    BUDGET = len(real_train)                             # постоянный бюджет: всегда 24 обучающих изображения
    N_REPEAT = 10
    MODELS = {'logreg': lambda: make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=3000)),
              'knn3': lambda: KNeighborsClassifier(n_neighbors=3, metric='cosine')}
    syn_pool = [i for i, y in enumerate(y_syn) if y in (2, 3)]        # общая поддержка классов с реальными изображениями (2 и 3)

    def make_train(alpha, seed):
        r = np.random.RandomState(seed); n_syn = int(round(alpha * BUDGET)); n_real = BUDGET - n_syn
        pick = lambda pool, y, n: np.concatenate([r.choice([i for i in pool if y[i] == c], n // 2, replace=False) for c in (2, 3)]) if n else np.array([], int)
        ir = pick(range(len(y_real_train)), y_real_train, n_real); is_ = pick(syn_pool, y_syn, n_syn)
        X = np.concatenate([F_real_train[ir], F_syn[is_]]); y = np.concatenate([y_real_train[ir], y_syn[is_]])
        return X, y, ir, is_

    trained, compositions = {}, []
    for alpha in ALPHAS:
        for seed in range(SEED, SEED + N_REPEAT):
            X_, y_, ir, is_ = make_train(alpha, seed)
            assert len(y_) == BUDGET and len(ir) + len(is_) == BUDGET and set(y_) == {2, 3}          # бюджет постоянен
            compositions.append({'alpha': alpha, 'seed': seed, 'n_real': int(len(ir)), 'n_synthetic': int(len(is_)),
                                 'real_ids': [real_train[i]['id'] for i in ir], 'synthetic_ids': [accepted[i]['id'] for i in is_]})
            for name, mk in MODELS.items(): trained[(alpha, seed, name)] = mk().fit(X_, y_)
    assert not ({c for comp in compositions for c in comp['real_ids']} & {r['id'] for r in real_test})      # test не участвует в обучении
    save_json('mixture_compositions.json', compositions)
    print('обучено моделей:', len(trained), '| α =', ALPHAS, '| повторов:', N_REPEAT, '| бюджет:', BUDGET)
else:
    print('FAST_DEV_RUN: смеси не обучаются.')

обучено моделей: 80 | α = [0.0, 0.25, 0.5, 1.0] | повторов: 10 | бюджет: 24


### Оценка на реальном тесте

*Вес: 1 балл.*

**Постановка.** Этап «оценка на реальном тесте» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Запишите предсказания и accuracy всех моделей только на реальном test.
2. Сохраните и предъявите mixture_results.json и predictions.jsonl.

**Результат.** mixture_results.json и predictions.jsonl.

**Критерий принятия.** Задание считается принятым, если mixture_results.json и predictions.jsonl получены из фактических входов этого этапа и доступны для проверки.

In [12]:
if RUN_FULL:
    rows = []
    for (alpha, seed, name), clf in trained.items():
        pred = clf.predict(F_real_test)                                       # оценка ТОЛЬКО на реальном test
        for r, p in zip(real_test, pred):
            rows.append({'alpha': alpha, 'seed': seed, 'model': name, 'id': r['id'], 'true': int(r['answer']), 'pred': int(p)})
    with (ARTIFACTS / 'predictions.jsonl').open('w') as f:
        for r in rows: f.write(json.dumps(r) + '\n')
    assert {r['id'] for r in rows} == {r['id'] for r in real_test} and len(rows) == len(trained) * len(real_test)

    test_ids = [r['id'] for r in real_test]; idx_of = {i: k for k, i in enumerate(test_ids)}
    corr = {}                                                                   # (alpha, model) -> [seeds x test]
    for r in rows: corr.setdefault((r['alpha'], r['model']), {}).setdefault(r['seed'], np.zeros(len(test_ids)))[idx_of[r['id']]] = float(r['pred'] == r['true'])
    corr = {k: np.stack([v[s] for s in sorted(v)]) for k, v in corr.items()}
    brs = np.random.RandomState(SEED); boots = [brs.randint(0, len(test_ids), len(test_ids)) for _ in range(2000)]
    results = {}
    for (alpha, name), M in corr.items():
        acc_seed = M.mean(1); base = corr[(0.0, name)].mean(0); cur = M.mean(0)
        d = [cur[b].mean() - base[b].mean() for b in boots]
        results.setdefault(str(alpha), {})[name] = {'accuracy_mean_over_seeds': float(acc_seed.mean()), 'accuracy_std_over_seeds': float(acc_seed.std(ddof=1)),
                                                    'n_runs': int(len(acc_seed)), 'n_test': len(test_ids),
                                                    'delta_vs_alpha0': float(cur.mean() - base.mean()),
                                                    'delta_bootstrap_95ci': [float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))]}
    best = {name: max(results, key=lambda a: results[a][name]['accuracy_mean_over_seeds']) for name in MODELS}
    best_syn = {name: max((a for a in results if a != '0.0'), key=lambda a: results[a][name]['accuracy_mean_over_seeds']) for name in MODELS}
    notes = []
    for name in MODELS:
        b = results[best_syn[name]][name]; ci = b['delta_bootstrap_95ci']
        notes.append(f"{name}: лучшая α по средней точности = {best[name]}; лучшая смесь с синтетикой α={best_syn[name]}: Δ к α=0 {b['delta_vs_alpha0']:+.3f}, "
                     f"95% CI [{ci[0]:+.3f}; {ci[1]:+.3f}] - " + ('различие не отличимо от нуля на 12 тестовых изображениях' if ci[0] <= 0 <= ci[1]
                     else ('синтетика улучшает перенос' if ci[0] > 0 else 'синтетика ухудшает перенос')))
    save_json('mixture_results.json', {'protocol': {'alphas': ALPHAS, 'budget': BUDGET, 'repeats': N_REPEAT, 'models': list(MODELS), 'test': 'real PixMo-Count test, n=12',
                                                    'chance_level': 0.5, 'synthetic_used': 'accepted, counts 2 and 3 only'}, 'results': results, 'notes': notes})
    print(f"{'α':>5s} " + ' '.join(f'{m:>22s}' for m in MODELS))
    for a in results: print(f'{a:>5s} ' + ' '.join(f"{results[a][m]['accuracy_mean_over_seeds']:.3f}±{results[a][m]['accuracy_std_over_seeds']:.3f} (Δ {results[a][m]['delta_vs_alpha0']:+.2f})".rjust(22) for m in MODELS))
    print('\n'.join(notes))
else:
    print('FAST_DEV_RUN: оценка на реальном test не выполняется.')

    α                 logreg                   knn3
  0.0  0.500±0.000 (Δ +0.00)  0.250±0.000 (Δ +0.00)
 0.25  0.450±0.153 (Δ -0.05)  0.350±0.123 (Δ +0.10)
  0.5  0.433±0.086 (Δ -0.07)  0.417±0.118 (Δ +0.17)
  1.0  0.467±0.090 (Δ -0.03)  0.458±0.098 (Δ +0.21)
logreg: лучшая α по средней точности = 0.0; лучшая смесь с синтетикой α=1.0: Δ к α=0 -0.033, 95% CI [-0.475; +0.409] - различие не отличимо от нуля на 12 тестовых изображениях
knn3: лучшая α по средней точности = 1.0; лучшая смесь с синтетикой α=1.0: Δ к α=0 +0.208, 95% CI [-0.125; +0.517] - различие не отличимо от нуля на 12 тестовых изображениях


### Творческое мини-исследование

*Бонус: до 1 балла сверх 10.*

**Постановка.** Сформулируйте проверяемую гипотезу о предметном ограничении метода данной лабораторной и проверьте её на дополнительных реальных входах без использования отложенного теста для выбора гипотезы. Сопоставьте результаты с основным экспериментом и объясните расхождения.

**Результат.** Отдельный файл artifacts/bonus.md с гипотезой, протоколом, наблюдениями и выводом.

**Критерий принятия.** Задание считается принятым, если гипотеза проверена по сохранённым предсказаниям или изображениям и приведено воспроизводимое сравнение.

## Анализ результатов

Сопоставьте измеренные показатели на фиксированной отложенной выборке, укажите ограничения выборки и разберите ошибки по сохранённым предсказаниям. FAST_DEV_RUN не является основанием для итогового вывода.

## Требования к сдаче

Запустите полный режим, приложите конфигурацию, версии зависимостей, артефакты и инструкции повторного запуска. Не сдавайте фиктивные измерения.

## Инструкция повторного запуска

1. Colab (GPU T4); файл `open_real_scenes.py` положите рядом с ноутбуком. Зависимости: `%pip install -q "transformers>=4.51,<5" accelerate bitsandbytes pillow torchvision scikit-learn scipy matplotlib`.
2. `FAST_DEV_RUN=True` проверяет только renderer, манифест, дедупликацию и инварианты (без VLM, без реальных изображений, без метрик). `FAST_DEV_RUN=False` — полный эксперимент: 240 синтетических сцен, 1 проход VLM, реальные PixMo-Count (24 train / 12 test), 4 доли синтетики × 10 повторов × 2 модели.
3. Артефакты пишутся в `artifacts/`; проверка VLM возобновляема (`verification.jsonl` дописывается построчно).
4. Реальный test состоит из 12 изображений: точность дискретна (шаг 1/12), поэтому выводы осторожные, а доверительные интервалы приведены в `mixture_results.json`.

In [13]:
import importlib.metadata as md, platform
def ver(n):
    try: return md.version(n)
    except Exception: return None
save_json('environment.json', {'python': platform.python_version(),
          'versions': {n: ver(n) for n in ['numpy', 'scikit-learn', 'scipy', 'pillow', 'torch', 'torchvision', 'transformers', 'bitsandbytes']},
          'config': {'SEED': SEED, 'FAST_DEV_RUN': FAST_DEV_RUN, 'N_SYN': N_SYN, 'space_size': len(SPACE), 'renderer_sha256': RENDERER_SHA256,
                     'vlm': globals().get('MODEL_ID')}})
print(sorted(p.name for p in ARTIFACTS.iterdir()))

['accept_stats.json', 'accepted.jsonl', 'dedup.json', 'environment.json', 'mixture_compositions.json', 'mixture_results.json', 'predictions.jsonl', 'real_manifest.jsonl', 'real_manifest_check.json', 'resnet18_features_lab12.npz', 'scene_specs.json', 'synthetic_manifest.jsonl', 'verification.jsonl']
